# Threshold sweep — finding LLMGPR's actual cut

**Settled so far.** LLMGPR's Foursquare is Section 5's *raw* dump: `raw_POIs.txt` restricted
to New York + Chicago + Los Angeles gives **exactly 436 categories**, their number, from a
519-category global vocabulary. The raw and filtered check-in files **share a user-id space**
(14,400 of 14,401 same id, vote purity 1.00), so friendships attach to raw check-ins directly.
And `dataset_WWW_Checkins` turns out to be exactly "raw check-ins of the friendship users"
(592,340 vs 592,341 in the same boxes).

**Disproven.** A social restriction cannot be their rule: only 14,400 friendship users are in
these three cities, and *all* of their check-ins come to 592,340 — 49% of the 1,214,631 needed,
before any filter. So LLMGPR did not restrict to socially-connected users.

**What their numbers actually look like** (New York, the cleanest signal): they keep **79% of
check-ins** and **56% of POIs** while dropping **93% of users**. That is not a k-core — a
k-core sheds check-ins in proportion. It is a pure **activity cut on users**, with POIs being
whatever the survivors happened to touch. Their 162 check-ins per user says the threshold is
far above 10.

So this notebook turns the knob we never turned. It sweeps the user threshold on its own,
then a full 2-D grid of (user threshold x POI threshold) in both single-pass and iterated
form, and ranks every rule by how well it reproduces all four columns at once.

Fast — it reuses `llmgpr_checkins_A.parquet` from the last run if you attach that notebook's
output as a data source. Add it via **+ Add Input -> Your Work -> the previous notebook**.
Failing that it re-derives from the zip, which costs the usual ~25 minutes.

## 0. Setup and load the 3-city check-ins

In [ ]:
import os, sys, re, zipfile, subprocess, itertools, gc
import pandas as pd, numpy as np

WORK="/kaggle/working"; os.makedirs(WORK, exist_ok=True)
TARGET_3CITY=dict(users=7_507, pois=80_962, cats=436, ck=1_214_631)
TARGET_NYC  =dict(users=6_078, pois=63_445, cats=436, ck=923_856)
CHUNK=2_000_000
CITY_BBOX={"New York":dict(lon_min=-74.3,lon_max=-73.6,lat_min=40.4,lat_max=41.0),
           "Chicago":dict(lon_min=-88.0,lon_max=-87.5,lat_min=41.6,lat_max=42.1),
           "Los Angeles":dict(lon_min=-118.7,lon_max=-117.6,lat_min=33.6,lat_max=34.4)}

def find(pat, roots=("/kaggle/input", WORK)):
    hits=[]
    for root in roots:
        if not os.path.isdir(root): continue
        for dp,_,fns in os.walk(root):
            if "__MACOSX" in dp: continue
            for fn in fns:
                if re.search(pat, fn, re.I) and not fn.startswith("._"):
                    hits.append(os.path.join(dp,fn))
    return sorted(hits)

def save_table(df, stem):
    try: df.to_parquet(f"{stem}.parquet", index=False); out=f"{stem}.parquet"
    except ImportError: df.to_csv(f"{stem}.csv", index=False); out=f"{stem}.csv"
    print("wrote", out); return out

def load(paths):
    p=paths[0]; print("loading", p)
    return pd.read_parquet(p) if p.endswith(".parquet") else pd.read_csv(p, dtype=str)

cached=find(r"llmgpr_checkins_A\.(parquet|csv)$")
if cached:
    ck=load(cached)
else:
    print("no cached parquet found - re-deriving from the zip (~25 min)")
    ZIP=f"{WORK}/dataset_WWW2019.zip"
    if not find(r"raw_Checkins.*\.txt$"):
        url=("https://drive.usercontent.google.com/download?"
             "id=1PNk3zY8NjLcDiAbzjABzY5FiPAFHq6T8&export=download&confirm=t")
        assert subprocess.run(f'curl -L --fail -o "{ZIP}" "{url}"',shell=True).returncode==0
        with zipfile.ZipFile(ZIP) as z:
            for n in z.namelist():
                if re.search(r"raw_(POIs|Checkins)", n) and "__MACOSX" not in n:
                    z.extract(n, WORK); print("  done", n, flush=True)
        os.remove(ZIP)
    RP=find(r"raw_POIs\.txt$")[0]; RC=find(r"raw_Checkins.*\.txt$")[0]
    cv={c:set() for c in CITY_BBOX}; V2C={}; V2CITY={}
    for chk in pd.read_csv(RP, sep="\t", header=None,
                           names=["venue_id","lat","lon","category","country"],
                           dtype={"venue_id":str,"category":str}, on_bad_lines="skip", chunksize=CHUNK):
        chk["lat"]=pd.to_numeric(chk["lat"],errors="coerce"); chk["lon"]=pd.to_numeric(chk["lon"],errors="coerce")
        chk=chk.dropna(subset=["lat","lon"])
        for city,b in CITY_BBOX.items():
            s=chk[chk["lon"].between(b["lon_min"],b["lon_max"]) & chk["lat"].between(b["lat_min"],b["lat_max"])]
            if len(s):
                cv[city]|=set(s["venue_id"]); V2C.update(zip(s["venue_id"],s["category"]))
                V2CITY.update({v:city for v in s["venue_id"]})
    keep=pd.Index(sorted(set().union(*cv.values()))); parts=[]
    for chk in pd.read_csv(RC, sep="\t", header=None, names=["user_id","venue_id","utc_time","tz_offset"],
                           dtype={"user_id":str,"venue_id":str,"utc_time":str},
                           usecols=[0,1,2,3], on_bad_lines="skip", chunksize=CHUNK):
        parts.append(chk[chk["venue_id"].isin(keep)])
    ck=pd.concat(parts,ignore_index=True); del parts; gc.collect()
    ck=ck.assign(city=ck["venue_id"].map(V2CITY), category=ck["venue_id"].map(V2C))
    save_table(ck, f"{WORK}/llmgpr_checkins_A")

print(f"\n{len(ck):,} check-ins | {ck['user_id'].nunique():,} users | "
      f"{ck['venue_id'].nunique():,} POIs | {ck['category'].nunique():,} categories")
print(ck.head(3).to_string())

## 1. How the activity is distributed

Before sweeping, look at the shape. Their table needs 4.9% of users to hold 54.5% of
check-ins in the three cities (6.8% holding 79% in New York). This says whether any
threshold can do that.

In [ ]:
def profile(d, label, target):
    n=len(d); uc=d["user_id"].value_counts()
    print(f"\n{label}: {n:,} check-ins over {len(uc):,} users")
    print(f"  their table wants {target['users']:,} users ({target['users']/len(uc):.1%}) "
          f"holding {target['ck']:,} ({target['ck']/n:.1%})")
    order=uc.sort_values(ascending=False).to_numpy()
    cum=np.cumsum(order)/n
    k=target["users"]
    if k<=len(order):
        print(f"  the top {k:,} users by activity actually hold {cum[k-1]:.1%} of check-ins "
              f"(threshold >= {order[k-1]:,} check-ins)")
    for pct in (0.5,0.6,0.7,0.79,0.9):
        i=int(np.searchsorted(cum,pct))+1
        print(f"    {pct:.0%} of check-ins sit in the top {i:,} users (>= {order[min(i,len(order))-1]:,} each)")

profile(ck, "ALL 3 CITIES", TARGET_3CITY)
profile(ck[ck["city"].eq("New York")], "NEW YORK", TARGET_NYC)

## 2. User-threshold sweep (no POI filter)

In [ ]:
def stats(d):
    return (d["user_id"].nunique(), d["venue_id"].nunique(),
            d["category"].nunique(), len(d))

def match(s, t):
    got=dict(zip(("users","pois","cats","ck"), s))
    return float(np.mean([min(got[k],t[k])/max(got[k],t[k]) for k in ("users","pois","cats","ck")]))

def sweep_users(d, target, label):
    uc=d["user_id"].value_counts()
    print(f"\n### {label}")
    hdr=f"{'>=Tu':>6}{'users':>10}{'POIs':>10}{'cats':>6}{'check-ins':>12}{'ck/user':>9}{'match':>8}"
    print(hdr); print("-"*len(hdr))
    best=(0,None)
    for Tu in (1,2,3,5,8,10,15,20,25,30,35,40,50,60,75,100,125,150,200):
        sub=d[d["user_id"].isin(uc[uc>=Tu].index)]
        if sub.empty: continue
        s=stats(sub); m=match(s,target)
        print(f"{Tu:>6}{s[0]:>10,}{s[1]:>10,}{s[2]:>6}{s[3]:>12,}{s[3]/s[0]:>9.1f}{m:>8.2f}")
        if m>best[0]: best=(m,Tu)
    print("-"*len(hdr))
    print(f"{'TARGET':>6}{target['users']:>10,}{target['pois']:>10,}{target['cats']:>6}"
          f"{target['ck']:>12,}{target['ck']/target['users']:>9.1f}{1.00:>8.2f}")
    print(f"best user-only threshold: Tu={best[1]}  (match {best[0]:.2f})")
    return best

best3 = sweep_users(ck, TARGET_3CITY, "ALL 3 CITIES vs CIKM table")
best1 = sweep_users(ck[ck["city"].eq("New York")], TARGET_NYC, "NEW YORK vs arXiv v1 table")

## 3. Full grid — user threshold x POI threshold, single-pass and iterated

"Users and POIs with less than 10 interactions are removed" has four degrees of freedom we
have never varied together: the two thresholds, and whether the removal is applied once or
iterated to a fixed point. This ranks every combination.

In [ ]:
TU=(1,2,3,5,8,10,15,20,25,30,40,50,60,75,100,150)
TP=(1,2,3,5,8,10)

def apply_rule(d, Tu, Tp, iterated):
    if not iterated:
        uc=d["user_id"].value_counts(); vc=d["venue_id"].value_counts()
        return d[d["user_id"].isin(uc[uc>=Tu].index) & d["venue_id"].isin(vc[vc>=Tp].index)]
    while True:
        n=len(d)
        uc=d["user_id"].value_counts(); d=d[d["user_id"].isin(uc[uc>=Tu].index)]
        vc=d["venue_id"].value_counts(); d=d[d["venue_id"].isin(vc[vc>=Tp].index)]
        if len(d)==n or d.empty: return d

def grid(d, target, label):
    rows=[]
    for i,(Tu,Tp,it) in enumerate(itertools.product(TU,TP,(False,True))):
        sub=apply_rule(d, Tu, Tp, it)
        if sub.empty: continue
        s=stats(sub); rows.append((match(s,target), Tu, Tp, it)+s)
        print(f"\r{label}: {i+1}/{len(TU)*len(TP)*2}", end="", flush=True)
    rows.sort(reverse=True)
    print(f"\n\n### {label} - top 15 of {len(rows)} rules")
    hdr=(f"{'match':>7}{'Tu':>5}{'Tp':>4}{'iter':>6}{'users':>10}{'POIs':>10}"
         f"{'cats':>6}{'check-ins':>12}{'ck/user':>9}")
    print(hdr); print("-"*len(hdr))
    for m,Tu,Tp,it,u,p,c,n in rows[:15]:
        print(f"{m:>7.3f}{Tu:>5}{Tp:>4}{str(it):>6}{u:>10,}{p:>10,}{c:>6}{n:>12,}{n/u:>9.1f}")
    print("-"*len(hdr))
    print(f"{'TARGET':>7}{'':>5}{'':>4}{'':>6}{target['users']:>10,}{target['pois']:>10,}"
          f"{target['cats']:>6}{target['ck']:>12,}{target['ck']/target['users']:>9.1f}")
    return rows

rows3 = grid(ck, TARGET_3CITY, "3 cities")
rows1 = grid(ck[ck["city"].eq("New York")], TARGET_NYC, "New York")

## 4. Verdict and emit

In [ ]:
m,Tu,Tp,it,u,p,c,n = rows3[0]
print(f"best 3-city rule: users >= {Tu}, POIs >= {Tp}, iterated={it}  -> match {m:.3f}")
print(f"  {u:,} users | {p:,} POIs | {c} categories | {n:,} check-ins | {n/u:.1f} per user")
t=TARGET_3CITY
for k,got,want in (("users",u,t['users']),("POIs",p,t['pois']),("cats",c,t['cats']),("check-ins",n,t['ck'])):
    print(f"    {k:<10}{got:>12,} vs {want:>12,}   {got/want:.2f}x")

if m >= 0.85:
    print("\n=> RULE FOUND. This is LLMGPR's preprocessing; the dataset below is theirs.")
elif m >= 0.70:
    print("\n=> CLOSE BUT NOT EXACT. Adopt this rule, report our own numbers, footnote the gap.")
else:
    print("\n=> NO RULE REPRODUCES THEIR TABLE even from the right dump. Their Table 1 is not")
    print("   derivable from any single filter over the source we have identified. Adopt the")
    print("   best rule, report our numbers, and state this in the paper - it is a real finding.")

best = apply_rule(ck, Tu, Tp, it)
save_table(best, f"{WORK}/llmgpr_final_checkins")
users=set(best["user_id"].unique())

# accept the parquet/csv emitted by llmgpr-rule-sweep, or the raw tsv from the zip
# "_final_" files are THIS notebook's own output from a previous run - skip them, or a
# re-run would filter an already-filtered edge set and understate coverage.
fo = ([f for f in find(r"llmgpr.*friendship_old\.(parquet|csv)$") if "_final_" not in f]
      or find(r"friendship_old.*\.(txt|tsv)$"))
if fo:
    f0 = fo[0]; print("edges from", f0)
    if f0.endswith(".parquet"):   e = pd.read_parquet(f0)
    elif f0.endswith(".csv"):     e = pd.read_csv(f0, dtype=str)
    else:                         e = pd.read_csv(f0, sep="\t", header=None, dtype=str)
    e = e.iloc[:, :2].astype(str); e.columns = ["u", "v"]
    e = e[e["u"].isin(users) & e["v"].isin(users)]     # shared id space: no mapping needed
    save_table(e, f"{WORK}/llmgpr_final_friendship_old")
    covered=len(users & (set(e["u"])|set(e["v"])))
    print(f"\nfriendship_old edges among retained users: {len(e):,}")
    print(f"users with >=1 edge: {covered:,} / {len(users):,} ({covered/len(users):.1%})"
          "   <- this is the ceiling on group construction")
else:
    print("\nNO EDGE FILE FOUND - attach llmgpr-rule-sweep's output (it saved\n  llmgpr_friendship_old.parquet) or the raw zip, then re-run this cell.")

pois=(best[["venue_id","city","category"]].drop_duplicates("venue_id"))
save_table(pois, f"{WORK}/llmgpr_final_pois")
print(f"\nwrote llmgpr_final_checkins / _pois / _friendship_old")

## What to send back

The two grid tables from section 3 and the verdict from section 4.

The `users with >=1 edge` line at the very end matters more than it looks: it is the hard
ceiling on how many groups can exist, since a group needs co-presence **and** a social tie.
If it is small, that constrains group construction no matter which rule wins — and it is the
number to check before writing any group code.